In [ ]:
# clone into the repo
# install requirements
# start worker (worker gets gpu status, loads weights to file, gets cloudflare url, registers itself)
# is able to take requests
!git clone https://github.com/prava241/vllm-router.git
%cd vllm-router

Cloning into 'vllm-router'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 82 (delta 35), reused 67 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 43.62 KiB | 744.00 KiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/vllm-router
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 58.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 55.2 MB/

/content/vllm-router/src/worker
--2026-08-13 21:12:18--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.8.1/cloudflared-linux-amd64 [following]
--2026-08-13 21:12:18--  https://github.com/cloudflare/cloudflared/releases/download/2026.8.1/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/136c5e62-bfff-44b1-b506-3d66ac6df4b8?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-08-13T22%3A05%3A43Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b1

In [ ]:
!python -m pip install --upgrade pip

!python -m pip install \
    torch==2.7.1 \
    torchvision==0.22.1 \
    torchaudio==2.7.1 \
    --index-url https://download.pytorch.org/whl/cu128

!python -m pip install -r requirements.txt


In [ ]:
# Download cloudflared into src/worker, since that's where worker.py runs from
%cd src/worker
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
%cd ../..

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    quantization="awq",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    dtype="half",
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=50,
)

outputs = llm.generate(
    ["Explain what a GPU is in one sentence."],
    sampling_params,
)

print(outputs[0].outputs[0].text)


In [13]:
# pull latest fixes from the dev branch (push your local changes to dev first)
!git pull origin main

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 468 bytes | 468.00 KiB/s, done.
From https://github.com/prava241/vllm-router
 * branch            main       -> FETCH_HEAD
   e2fe516..5d92dcc  main       -> origin/main
Updating e2fe516..5d92dcc
Fast-forward
 src/worker/worker.py | 3 +++
 1 file changed, 3 insertions(+)


Before running the next cell: on your local machine (next to `server.py`), start the controller and expose it publicly so this Colab worker can reach it:

```
uvicorn src.server:app --host 0.0.0.0 --port 8000
./cloudflared tunnel --url http://localhost:8000
```

Copy the printed `https://*.trycloudflare.com` URL and paste it into `CONTROLLER_URL` below.

In [6]:
CONTROLLER_URL = "https://litigation-drama-regression-therefore.trycloudflare.com"  # replace with your current tunnel URL
!ls

!python -m worker.worker --controller-url {CONTROLLER_URL} --host 0.0.0.0 --port 8000

controller.py  models.py  policies.py  __pycache__  server.py  worker
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/_ops.py", line 1503, in load_library
    ctypes.CDLL(path)
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvrtc.so.13: cannot open shared object file: No such file or directory

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/c